## 3.5 预定义状态
### 3.5.1. MessagesState

`LangGraph` 构建的计算图通常会和 `LLM` 结合使用，而 `LLM` 在运行过程中通常需要维护一组消息列表。为了提升开发效率，`LangGraph` 官方提供了一个预定义状态类型：`langgraph.graph.message.MessagesState`。

我们可以通过继承该状态类型，以在其基础上扩展自定义状态字段。

```python
class MessagesState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

add_messages(left, right, *, format=None)
```

示例:

In [1]:
from langchain_deepseek import ChatDeepSeek
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import MessagesState
from langchain.messages import HumanMessage
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

# 从.env文件中加载环境变量
load_dotenv(override=True, verbose=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
OPENAI_API_BASE = os.environ["OPENAI_API_BASE"]  # 与本项目 .env 中的名称一致

model = ChatOpenAI(
    model="gpt-6-luna",  # 当前配置已验证可用；原 gpt-5.5 返回模型权限 403
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_API_BASE,
)

# 继承 Messages状态
class OverAllState(MessagesState):
    username: str
    output: str

def node_a(state: OverAllState) -> OverAllState:
    return {
        "messages": [HumanMessage("你好，我是 " + state["username"])]
    }

def llm_node(state: OverAllState) -> OverAllState:
    res = model.invoke(state["messages"])

    return {
        "messages": [res],
        "output": res.content
    }

builder = StateGraph(state_schema=OverAllState)
builder.add_node("node_a", node_a)
builder.add_node("llm_node", llm_node)
builder.add_edge(START, "node_a")
builder.add_edge("node_a", "llm_node")
builder.add_edge("llm_node", END)

graph = builder.compile()

response = graph.invoke({"username":"小李"})
from rich import print as rprint

rprint(response)


{
    'messages': [
        HumanMessage(
            content='你好，我是 小李',
            additional_kwargs={},
            response_metadata={},
            id='d10fa9b9-0641-4d4e-9df8-b5a941b8da53'
        ),
        AIMessage(
            content='你好，小李。有什么我可以帮你处理的？',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 16,
                    'prompt_tokens': 4391,
                    'total_tokens': 4407,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': None,
                        'text_tokens': 0,
                        'image_tokens': 0
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cache_write_tokens': None,
                        'cached_tokens': 3840,
                        'image_tokens': 0,
                        'text_tokens': 0
                    },
                    'billing_usage': {
                        'source': 'oai_responses',
                        'semantic': 'openai',
                        'openai_usage': {
                            'prompt_tokens': 0,
                            'completion_tokens': 0,
                            'total_tokens': 4407,
                            'prompt_tokens_details': {
                                'cached_tokens': 0,
                                'text_tokens': 0,
                                'audio_tokens': 0,
                                'image_tokens': 0
                            },
                            'completion_tokens_details': {
                                'text_tokens': 0,
                                'audio_tokens': 0,
                                'image_tokens': 0,
                                'reasoning_tokens': 0
                            },
                            'input_tokens': 4391,
                            'output_tokens': 16,
                            'input_tokens_details': {
                                'cached_tokens': 3840,
                                'text_tokens': 0,
                                'audio_tokens': 0,
                                'image_tokens': 0
                            },
                            'output_tokens_details': {
                                'text_tokens': 0,
                                'audio_tokens': 0,
                                'image_tokens': 0,
                                'reasoning_tokens': 0
                            },
                            'claude_cache_creation_5_m_tokens': 0,
                            'claude_cache_creation_1_h_tokens': 0
                        }
                    },
                    'input_tokens': 4391,
                    'output_tokens': 16,
                    'input_tokens_details': None,
                    'claude_cache_creation_5_m_tokens': 0,
                    'claude_cache_creation_1_h_tokens': 0
                },
                'model_provider': 'openai',
                'model_name': 'gpt-6-luna',
                'system_fingerprint': None,
                'id': 'chatcmpl-202610140616302908623978268d9d6bp26Mpq0',
                'service_tier': 'default',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--01a1390e-1f7e-7d22-9956-cbb8823a9f00-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 4391,
                'output_tokens': 16,
                'total_tokens': 4407,
                'input_token_details': {'audio': 0, 'cache_read': 3840},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        )
    ]

In [2]:
print(type(response))
# 打印出来说字典,但是像overAllState

<class 'dict'>


只关注 `messages` 状态，执行流程如下：

1. `node_a` 返回一条 `HumanMessage`；
2. `LangGraph` 使用 `add_messages` 将该消息合并到 `messages` 状态字段中；
3. `llm_node` 从 `state["messages"]` 中读取完整消息列表，并调用模型；
4. `llm_node` 将模型生成的 `AIMessage` 作为状态更新返回；
5. `LangGraph` 再次通过 `add_messages` 将 `AIMessage` 合并到 `messages` 中；
6. 最终状态中包含完整消息列表。

### 3.5.2. AgentState
这个在 隔壁 langchain 项目仓库里做过一些笔记。第一次使用是在自定义中间件的输入参数。

源码定义如下(只包括重要部分):
```python
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]   # 必有：消息列表，合并规则是 add_messages
    structured_response: NotRequired[...]                  # 可选：结构化输出的结果
    jump_to: NotRequired[...]                              # 可选：框架内部使用的跳转标记（见 5.3.4）
```


中间件定义如下，而 `runtime: Runtime` 是运行时对象，context_schema 只决定其中 runtime.context 的类型。Runtime 里还有 store、stream_writer 等

```python
from langchain.agents.middleware import AgentMiddleware

class MyMiddleware(AgentMiddleware):
    def before_model(self,state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        state["messages"][-1].content += "----> before_model <-----"
        return None
```

在之前的 [中间件学习笔记里](/Users/skk/Developer/lang-chain-learn/Chapter08_Middleware/08-自定义中间件-jump.ipynb)，agentState 的字段 和 MessageState 一样需要先继承再扩展。区别是扩展的方式不同。


需要先扩展 state 的结构：

1. 定义一个继承 `AgentState` 的类，加上自定义字段。字段一般用 `NotRequired` 包起来，表示“一开始可以没有这个键”，读取时用 `state.get(键, 默认值)`。
2. 通过 `state_schema=` 交给装饰器。`create_agent` 会把所有中间件的 `state_schema` 合并成 Agent 的 state 结构。

create_agent 的图不是自己建的，所以要通过中间件的 state_schema，或者 create_agent(state_schema=...) 来扩展。

In [3]:
from typing import Any
from typing_extensions import NotRequired
from langchain.agents.middleware import AgentState, after_model, before_model
from langchain.messages import AIMessage, HumanMessage
from langgraph.runtime import Runtime

class CallCountState(AgentState):
    """在 AgentState 的基础上增加一个自定义字段"""
    model_call_count: NotRequired[int]   # NotRequired：一开始可以没有这个键

@before_model(state_schema=CallCountState, can_jump_to=["end"])
def check_call_limit(state: CallCountState, runtime: Runtime) -> dict[str, Any] | None:
    count = state.get("model_call_count", 0)          # 读取自定义字段，没有时按 0 算
    print(f"[check_call_limit] 已调用模型 {count} 次")
    if count >= 3:
        return {
            "messages": [AIMessage(f"模型调用次数已达上限（{MAX_CALLS} 次），本次不再调用模型。")],
            "jump_to": "end",
        }
    return None

该状态中主要包含三个字段。

1. messages

```
messages: Required[Annotated[list[AnyMessage], add_messages]]
```

`messages` 用于存储 Agent 运行过程中的消息列表。

该字段和 `MessagesState` 中的 `messages` 字段类似，也使用 `add_messages` 作为 `Reducer`。

2. jump_to

```
jump_to: NotRequired[Annotated[JumpTo | None, EphemeralValue, PrivateStateAttr]]
```

`jump_to` 是 `LangChain Agent` 内部使用的控制字段，主要服务于 Agent 中间件体系。

在自定义 `StateGraph` 中，即使状态中定义了 `jump_to` 字段，`LangGraph` 也不会因为该字段的值自动跳转到某个节点。普通 `LangGraph` 运行图如果需要控制后续流向，通常应使用：

```python
Command(goto="node_name")
```


这里的Command表示图“接下来怎么做”。它有四个字段：

| 字段 | 作用 |
|---|---|
| `resume` | 对暂停的回复，也就是 `interrupt()` 恢复时的返回值。可以是单个值，也可以是 `{中断id: 回复}` 字典 |
| `update` | 要写入 state 的更新，作用和节点返回的字典相同 |
| `goto` | 接下来跳到哪个（或哪些）节点 |
| `graph` | 指令发给哪个图：默认是当前图，`Command.PARENT` 表示父图（用于子图） |



In [4]:
import operator
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command


class PathState(TypedDict):
    path: Annotated[list[str], operator.add]  # 记录经过的节点


def node_a(state: PathState):
    return Command(update={"path": ["a"]}, goto="node_c")  # 更新 state，并直接跳到 node_c


def node_b(state: PathState):
    return {"path": ["b"]}


def node_c(state: PathState):
    return {"path": ["c"]}


path_builder = StateGraph(PathState)
path_builder.add_node("node_a", node_a)
path_builder.add_node("node_b", node_b)
path_builder.add_node("node_c", node_c)
path_builder.add_edge(START, "node_a")
path_builder.add_edge("node_b", "node_c")
path_builder.add_edge("node_c", END)
print(path_builder.compile().invoke({"path": []}))

{'path': ['a', 'c']}


3. structured_response

```python
structured_response: NotRequired[Annotated[ResponseT, OmitFromInput]]
```

`structured_response` 用于存储 Agent 最终生成的结构化输出。

主要用于 agent 预定义的类输出。

# 总结
1. 一般不推荐使用 `agentstate` 扩展图状态，而是推荐使用 `MessageState`。`agentstate` 是langchain框架在 图上做的扩展。

# 问题
目前没啥问题